# Lab 11 — Embeddings and Vector Search

This notebook builds a full semantic search system on top of the ArXiv paper dataset cleaned in Lab 9/10.  
It covers embedding generation, cosine/dot/Euclidean similarity, ChromaDB setup, semantic search, keyword search, hybrid search (RRF), and analytical questions.

## Part 1 — Imports and Setup

In [2]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.embeddings.embedder     import get_model, generate_embeddings, embed_dataframe, make_paper_text
from src.embeddings.chroma_store import get_client, get_or_create_collection, add_papers, query_collection
from src.embeddings.search_engine import semantic_search, keyword_search, compare_search
from src.embeddings.hybrid_search  import hybrid_search, reciprocal_rank_fusion

CLEANED_CSV = Path('../data/processed/cleaned/cleaned_data.csv')

df = pd.read_csv(CLEANED_CSV)
print(f'Dataset shape: {df.shape}')
df.head(3)

Dataset shape: (50, 12)


,paper_id,title,authors,abstract,published_date,primary_category,all_categories,pdf_url,citation_count,relevance_score,language,published_year
0,2605.00924,Styleshield: Exposing The Fragility Of Aigc De...,Guantian Zheng,AI-generated content (AIGC) detectors are incr...,2026-04-30,cs.LG,cs.LG|cs.AI,https://arxiv.org/pdf/2605.00924,81,0.5557,en,2026
1,2604.26990,Ucsc-Nlp At Semeval-2026 Task 13: Multi-View G...,Kargi Chauhan; Sadiba Nusrat Nur,With the rapid growth of large language models...,2026-04-28,cs.SE,cs.SE,https://arxiv.org/pdf/2604.26990,94,0.6375,en,2026
2,2604.25370,Gpt-Image-2 In The Wild: A Twitter Dataset Of ...,Kidus Zewde; Simiao Ren; Xingyu Shen; Jenny Wu...,The release of GPT-image-2 by OpenAI marks a w...,2026-04-28,cs.CV,cs.CV|cs.AI,https://arxiv.org/pdf/2604.25370,28,0.5698,en,2026


## Part 2 — Understanding Embeddings

An embedding is a fixed-length list of floating-point numbers that represents the *meaning* of a text.  
We use the `all-MiniLM-L6-v2` model from HuggingFace, which produces 384-dimensional vectors.  
Texts with similar meaning will have vectors that are close together in that 384-dimensional space.

In [3]:
# Load model (downloaded ~90 MB on first run)
model = get_model()
print(f'Model: {model}')
print(f'Embedding dimension: {model.get_sentence_embedding_dimension()}')

2026-05-13 11:20:28,845 - INFO - [Embedder] Loading model: all-MiniLM-L6-v2
2026-05-13 11:20:28,853 - INFO - No device provided, using cpu
2026-05-13 11:20:29,294 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-13 11:20:29,337 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-05-13 11:20:29,616 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-13 11:20:29,663 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-13 11:20:29,669 - INFO - Loading Sentence

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-05-13 11:20:31,634 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-05-13 11:20:31,842 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-13 11:20:32,058 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-13 11:20:32,250 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-13 11:20:32,558 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-13 11:20:32,575 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/senten

Model: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)
Embedding dimension: 384


/tmp/ipykernel_12887/2584244258.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Embedding dimension: {model.get_sentence_embedding_dimension()}')


In [4]:
# Sample texts to demonstrate embedding generation
sample_texts = [
    'Deep learning model detects AI-generated text with high accuracy.',
    'Neural network identifies machine-written content in academic papers.',
    'Transformer architecture achieves state-of-the-art NLP performance.',
    'Reinforcement learning agent plays chess at grandmaster level.',
    'Romance novel explores love and loss in 19th century Paris.',
]

sample_embeddings = generate_embeddings(sample_texts)
print(f'Shape: {sample_embeddings.shape}')   # expected: (5, 384)
print(f'First vector (first 8 values): {sample_embeddings[0, :8]}')

2026-05-13 11:20:38,549 - INFO - [Embedder] Generated embeddings: shape=(5, 384)


Shape: (5, 384)
First vector (first 8 values): [-0.05711742 -0.04008681  0.00753845  0.06221333  0.0176378  -0.01133799
 -0.01771473 -0.10895433]


**Observation:** The output shape is `(5, 384)` — five texts, each represented as 384 numbers.  
No matter how long or short the original sentence, the model always produces a vector of exactly 384 values.

### 2.1 Why Keyword Search Fails

Keyword search only finds exact word matches. Searching for *"AI text detection"* misses papers about  
*"machine-generated content identification"* — same concept, different words.  
Embeddings solve this by encoding **meaning**, not exact words.

## Part 3 — Computing Similarity Between Texts

| Measure | Range | Interpretation |
|---|---|---|
| Cosine similarity | -1 to 1 | 1 = identical meaning, 0 = unrelated |
| Dot product | -∞ to +∞ | Equivalent to cosine when vectors are normalised |
| Euclidean distance | 0 to +∞ | 0 = identical, larger = more different |

In [5]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import euclidean

# Cosine similarity matrix for all sample texts
cos_matrix = cosine_similarity(sample_embeddings)
cos_df = pd.DataFrame(cos_matrix,
    index=[f'T{i+1}' for i in range(5)],
    columns=[f'T{i+1}' for i in range(5)])

print('Cosine Similarity Matrix:')
print(cos_df.round(4))

print('\nText labels:')
for i, t in enumerate(sample_texts):
    print(f'  T{i+1}: {t[:60]}...')

Cosine Similarity Matrix:
        T1      T2      T3      T4      T5
T1  1.0000  0.4977  0.4055  0.1912 -0.0002
T2  0.4977  1.0000  0.2882  0.1343  0.0940
T3  0.4055  0.2882  1.0000  0.0467 -0.0054
T4  0.1912  0.1343  0.0467  1.0000 -0.0741
T5 -0.0002  0.0940 -0.0054 -0.0741  1.0000

Text labels:
  T1: Deep learning model detects AI-generated text with high accu...
  T2: Neural network identifies machine-written content in academi...
  T3: Transformer architecture achieves state-of-the-art NLP perfo...
  T4: Reinforcement learning agent plays chess at grandmaster leve...
  T5: Romance novel explores love and loss in 19th century Paris....


**Observation:** T1 and T2 (both about AI text detection) have a high similarity score.  
T5 (romance novel) scores near 0 against all AI/ML texts — it is semantically unrelated.

In [6]:
# Semantic search using raw cosine similarity — query vs all samples
query_text = 'detecting machine-generated academic content'
query_emb  = generate_embeddings([query_text])  # shape (1, 384)

sims = cosine_similarity(query_emb, sample_embeddings)[0]
ranked = sorted(enumerate(sims), key=lambda x: x[1], reverse=True)

print(f'Query: "{query_text}"\n')
for rank, (idx, score) in enumerate(ranked, 1):
    print(f'  #{rank} (score={score:.4f}): {sample_texts[idx]}')

2026-05-13 11:20:50,624 - INFO - [Embedder] Generated embeddings: shape=(1, 384)


Query: "detecting machine-generated academic content"

  #1 (score=0.5963): Neural network identifies machine-written content in academic papers.
  #2 (score=0.3704): Deep learning model detects AI-generated text with high accuracy.
  #3 (score=0.1479): Transformer architecture achieves state-of-the-art NLP performance.
  #4 (score=0.0721): Reinforcement learning agent plays chess at grandmaster level.
  #5 (score=0.0054): Romance novel explores love and loss in 19th century Paris.


**Observation:** The query about *detecting machine-generated content* correctly ranks T1 and T2 highest,  
even though the exact phrase does not appear in either text.

In [7]:
# Compare all three similarity measures between text pairs
ai_text       = sample_embeddings[0]   # AI detection
ml_text       = sample_embeddings[2]   # Transformer / NLP
romance_text  = sample_embeddings[4]   # Romance novel

print('Comparing AI detection text vs ML text vs Romance text:')
print(f'  Cosine similarity  (AI vs ML):      {cosine_similarity([ai_text], [ml_text])[0][0]:.4f}')
print(f'  Cosine similarity  (AI vs Romance): {cosine_similarity([ai_text], [romance_text])[0][0]:.4f}')
print()
print(f'  Dot product        (AI vs ML):      {np.dot(ai_text, ml_text):.4f}')
print(f'  Dot product        (AI vs Romance): {np.dot(ai_text, romance_text):.4f}')
print()
print(f'  Euclidean distance (AI vs ML):      {euclidean(ai_text, ml_text):.4f}')
print(f'  Euclidean distance (AI vs Romance): {euclidean(ai_text, romance_text):.4f}')

Comparing AI detection text vs ML text vs Romance text:
  Cosine similarity  (AI vs ML):      0.4055
  Cosine similarity  (AI vs Romance): -0.0002

  Dot product        (AI vs ML):      0.4055
  Dot product        (AI vs Romance): -0.0002

  Euclidean distance (AI vs ML):      1.0904
  Euclidean distance (AI vs Romance): 1.4143


**Observation:**  
- **Cosine similarity** and **dot product** give *higher* values for more similar texts.  
- **Euclidean distance** gives *lower* values for more similar texts.  
- The AI detection text and the ML/NLP text are closer to each other than to the romance text across all three measures.

## Part 4 — Setting Up ChromaDB

ChromaDB is a persistent vector database. Unlike a pandas DataFrame (in-memory only),  
ChromaDB saves embeddings to disk so you only need to generate them once.  
Our collection is stored at `data/embeddings/chroma_db/` and uses cosine similarity.

In [8]:
client     = get_client()
collection = get_or_create_collection(client)
print(f'Collection: {collection.name}')
print(f'Documents in collection: {collection.count()}')

2026-05-13 11:20:57,282 - INFO - [ChromaStore] Collection 'papers' ready — 0 documents.


Collection: papers
Documents in collection: 0


## Part 5 — Adding Papers to ChromaDB

Each paper is stored as a document (title + category + abstract) with metadata  
(published_year, primary_category, language, citation_count, relevance_score).  
Papers already in the collection are skipped, so re-running is safe.

In [9]:
added = add_papers(collection, df)
print(f'New papers added: {added}')
print(f'Total papers in collection: {collection.count()}')

2026-05-13 11:21:11,794 - INFO - [ChromaStore] Embedding 50 new papers ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-13 11:21:15,288 - INFO - [Embedder] Generated embeddings: shape=(50, 384)
2026-05-13 11:21:15,461 - INFO - [ChromaStore] Upserted 50 papers.


New papers added: 50
Total papers in collection: 50


## Part 6 — Querying ChromaDB by Text Similarity

ChromaDB embeds the query automatically and returns the closest matching papers.  
We can send multiple queries in a single API call for efficiency.

In [10]:
# Multi-query call — two queries in one API call
multi_results = query_collection(
    collection,
    query_texts=[
        'large language model text generation',
        'image recognition convolutional neural network',
    ],
    n_results=3,
)

for res in multi_results:
    print(f"\nQuery: \"{res['query']}\"")
    for title, dist in zip(res['metadatas'], res['distances']):
        print(f"  [{1-dist:.3f}] {title['title'][:80]}")

2026-05-13 11:21:19,366 - INFO - [Embedder] Generated embeddings: shape=(2, 384)



Query: "large language model text generation"
  [0.499] C-Red: A Comprehensive Chinese Benchmark For Ai-Generated Text Detection Derived
  [0.481] Reasoning-Aware Aigc Detection Via Alignment And Reinforcement
  [0.479] Breaking The Generator Barrier: Disentangled Representation For Generalizable Ai

Query: "image recognition convolutional neural network"
  [0.339] Dymapia: A Multi-Domain Framework For Detecting Ai-Based Video Manipulation
  [0.309] From Redaction To Restoration: Deep Learning For Medical Image Anonymization And
  [0.307] Combating Pattern And Content Bias: Adversarial Feature Learning For Generalized


**Observation:** Both queries return semantically relevant papers even when the exact search terms  
may not appear verbatim in the paper titles. ChromaDB ranks by cosine distance to the query embedding.

## Part 7 — Metadata Filtering

ChromaDB supports combining semantic search with structured filters.  
We demonstrate four operators: `$eq`, `$gte`, `$in`, and `$and`.

In [11]:
# Filter 1: $eq — only English papers
results_en = query_collection(
    collection,
    query_texts=['deep learning classification'],
    n_results=3,
    where={'language': {'$eq': 'en'}},
)
print('Filter: language == "en"')
for m, d in zip(results_en[0]['metadatas'], results_en[0]['distances']):
    print(f"  [{1-d:.3f}] [{m['language']}] {m['title'][:70]}")

2026-05-13 11:21:22,818 - INFO - [Embedder] Generated embeddings: shape=(1, 384)


Filter: language == "en"
  [0.353] [en] From Redaction To Restoration: Deep Learning For Medical Image Anonymi
  [0.332] [en] Dymapia: A Multi-Domain Framework For Detecting Ai-Based Video Manipul
  [0.326] [en] Combating Pattern And Content Bias: Adversarial Feature Learning For G


In [12]:
# Filter 2: $gte — papers published in 2024 or later
results_recent = query_collection(
    collection,
    query_texts=['transformer self-attention mechanism'],
    n_results=3,
    where={'published_year': {'$gte': 2024}},
)
print('Filter: published_year >= 2024')
for m, d in zip(results_recent[0]['metadatas'], results_recent[0]['distances']):
    print(f"  [{1-d:.3f}] [{m['published_year']}] {m['title'][:70]}")

2026-05-13 11:21:24,918 - INFO - [Embedder] Generated embeddings: shape=(1, 384)


Filter: published_year >= 2024
  [0.317] [2026] Single-Cycle Multidirectional Eog Classification Faster Than Human Rea
  [0.236] [2026] Efficient Zero-Shot Ai-Generated Image Detection
  [0.177] [2026] Layer Consistency Matters: Elegant Latent Transition Discrepancy For G


In [13]:
# Filter 3: $in — papers in cs.LG or cs.AI categories
results_cat = query_collection(
    collection,
    query_texts=['reinforcement learning reward optimization'],
    n_results=3,
    where={'primary_category': {'$in': ['cs.LG', 'cs.AI']}},
)
print('Filter: primary_category in [cs.LG, cs.AI]')
for m, d in zip(results_cat[0]['metadatas'], results_cat[0]['distances']):
    print(f"  [{1-d:.3f}] [{m['primary_category']}] {m['title'][:70]}")

2026-05-13 11:21:27,786 - INFO - [Embedder] Generated embeddings: shape=(1, 384)


Filter: primary_category in [cs.LG, cs.AI]
  [0.226] [cs.AI] Alignment Imprint: Zero-Shot Ai-Generated Text Detection Via Provable 
  [0.208] [cs.AI] Reasoning-Aware Aigc Detection Via Alignment And Reinforcement
  [0.092] [cs.LG] Styleshield: Exposing The Fragility Of Aigc Detectors Through Continuo


In [14]:
# Filter 4: $and — cs.LG papers published >= 2025
results_combo = query_collection(
    collection,
    query_texts=['neural network training optimization'],
    n_results=3,
    where={
        '$and': [
            {'primary_category': {'$eq': 'cs.LG'}},
            {'published_year':   {'$gte': 2025}},
        ]
    },
)
print('Filter: primary_category == "cs.LG" AND published_year >= 2025')
for m, d in zip(results_combo[0]['metadatas'], results_combo[0]['distances']):
    print(f"  [{1-d:.3f}] [{m['primary_category']}] [{m['published_year']}] {m['title'][:60]}")

2026-05-13 11:21:30,087 - INFO - [Embedder] Generated embeddings: shape=(1, 384)


Filter: primary_category == "cs.LG" AND published_year >= 2025
  [0.192] [cs.LG] [2026] Styleshield: Exposing The Fragility Of Aigc Detectors Throug


**Observation:** Metadata filters allow us to narrow semantic search to specific subsets of the dataset.  
This is useful when the user knows a category or time range and wants to restrict results.  
The `$and` operator is particularly powerful for combining multiple constraints.

## Part 8 — Building the Search Engine Module

The `search_engine.py` module wraps ChromaDB with three clean functions:  
- `semantic_search()` — meaning-based retrieval via ChromaDB  
- `keyword_search()` — exact word matching across title and abstract  
- `compare_search()` — runs both and prints results side by side

In [15]:
# Semantic search
sem_results = semantic_search(collection, 'AI-generated text detection', n_results=5)
print('Semantic search results:')
print(sem_results.to_string(index=False))

2026-05-13 11:21:32,772 - INFO - [Embedder] Generated embeddings: shape=(1, 384)
2026-05-13 11:21:32,788 - INFO - [SearchEngine] Semantic search 'AI-generated text detection' → 5 results


Semantic search results:
  paper_id                                                                                                                         title primary_category  published_year  citation_count  similarity
2604.11796                      C-Red: A Comprehensive Chinese Benchmark For Ai-Generated Text Detection Derived From Real-World Prompts            cs.CL            2026              45      0.5630
2604.13692                               Breaking The Generator Barrier: Disentangled Representation For Generalizable Ai-Text Detection            cs.CL            2026              35      0.5098
2604.03437 Is It Cake Or Is It Ai? A Systematic Review Of Human Uncertainty In Distinguishing Generative Artificial Intelligence Content          stat.AP            2026              35      0.5048
2603.19316          Rising Prevalence Of Detected Ai-Generated Text In Medical Literature: Longitudinal Analysis In Open Access Articles            cs.DL            2026              

In [16]:
# Keyword search
kw_results = keyword_search(df, 'AI-generated text detection', n_results=5)
print('Keyword search results:')
if kw_results.empty:
    print('  (no exact matches found)')
else:
    print(kw_results.to_string(index=False))

2026-05-13 11:21:36,258 - INFO - [SearchEngine] Keyword search 'AI-generated text detection' → 3 results


Keyword search results:
  paper_id                                                                                                                 title primary_category  published_year  citation_count
2604.21300 Explainable Disentangled Representation Learning For Generalizable Authorship Attribution In The Era Of Generative Ai            cs.CL            2026               3
2604.16923                          Alignment Imprint: Zero-Shot Ai-Generated Text Detection Via Provable Preference Discrepancy            cs.AI            2026             103
2604.11796              C-Red: A Comprehensive Chinese Benchmark For Ai-Generated Text Detection Derived From Real-World Prompts            cs.CL            2026              45


In [17]:
# Side-by-side comparison
compare_search(collection, df, 'machine learning for natural language processing')

2026-05-13 11:21:39,386 - INFO - [SearchEngine] Keyword search 'machine learning for natural language processing' → 0 results
2026-05-13 11:21:39,423 - INFO - [Embedder] Generated embeddings: shape=(1, 384)
2026-05-13 11:21:39,438 - INFO - [SearchEngine] Semantic search 'machine learning for natural language processing' → 5 results



Query: "machine learning for natural language processing"

--- Keyword Search (5 results) ---
  (no exact keyword matches)

--- Semantic Search (5 results) ---
  paper_id                                                                                                                   title primary_category  published_year  citation_count  similarity
 2604.2699 Ucsc-Nlp At Semeval-2026 Task 13: Multi-View Generalization And Diagnostic Analysis Of Machine-Generated Code Detection            cs.SE            2026              94      0.3919
2604.11796                C-Red: A Comprehensive Chinese Benchmark For Ai-Generated Text Detection Derived From Real-World Prompts            cs.CL            2026              45      0.3400
2604.19172                                                          Reasoning-Aware Aigc Detection Via Alignment And Reinforcement            cs.AI            2026              91      0.3363
2604.09514                              Many Ways To Be Fake: Benchmark

## Part 9 — Complete Semantic Search System

Testing several natural language queries end-to-end, comparing keyword vs semantic results.

In [18]:
demo_queries = [
    'detecting fake content written by computers',
    'training large language models efficiently',
    'computer vision object detection',
]

for q in demo_queries:
    compare_search(collection, df, q, n_results=4)

2026-05-13 11:21:42,904 - INFO - [SearchEngine] Keyword search 'detecting fake content written by computers' → 0 results
2026-05-13 11:21:42,938 - INFO - [Embedder] Generated embeddings: shape=(1, 384)
2026-05-13 11:21:42,956 - INFO - [SearchEngine] Semantic search 'detecting fake content written by computers' → 4 results
2026-05-13 11:21:42,976 - INFO - [SearchEngine] Keyword search 'training large language models efficiently' → 0 results
2026-05-13 11:21:43,029 - INFO - [Embedder] Generated embeddings: shape=(1, 384)
2026-05-13 11:21:43,047 - INFO - [SearchEngine] Semantic search 'training large language models efficiently' → 4 results
2026-05-13 11:21:43,065 - INFO - [SearchEngine] Keyword search 'computer vision object detection' → 0 results



Query: "detecting fake content written by computers"

--- Keyword Search (4 results) ---
  (no exact keyword matches)

--- Semantic Search (4 results) ---
  paper_id                                                                                                                 title primary_category  published_year  citation_count  similarity
2604.12216                      Timemark: A Trustworthy Time Watermarking Framework For Exact Generation-Time Recovery From Aigc            cs.CR            2026              11      0.4782
2604.09514                            Many Ways To Be Fake: Benchmarking Fake News Detection Under Strategy-Driven Ai Generation            cs.CL            2026              70      0.4557
2604.22089                                                Ethics Testing: Proactive Identification Of Generative Ai System Harms            cs.SE            2026              29      0.4424
 2604.1046 Toward Accountable Ai-Generated Content On Social Platforms: Steganograph

2026-05-13 11:21:43,188 - INFO - [Embedder] Generated embeddings: shape=(1, 384)
2026-05-13 11:21:43,205 - INFO - [SearchEngine] Semantic search 'computer vision object detection' → 4 results


  paper_id                                                                                                                     title primary_category  published_year  citation_count  similarity
2604.24426                                               Dymapia: A Multi-Domain Framework For Detecting Ai-Based Video Manipulation            cs.CV            2026              13      0.2985
2604.24004 Single-Cycle Multidirectional Eog Classification Faster Than Human Reaction Time For Wearable Human-Computer Interactions          eess.SP            2026             114      0.2868
2603.28508                                  Generalizable Detection Of Ai Generated Images With Large Models And Fuzzy Decision Tree            cs.CV            2026             106      0.2807
2605.00874                                               Latent Space Probing For Adult Content Detection In Video Generative Models            cs.CV            2026              75      0.2693



### Analysis: When does each search method work better?

**Semantic search performs better when:**
- The query uses different words than the document (synonyms, paraphrases)
  - e.g. *'detecting fake content written by computers'* finds papers about AI-generated text detection even without those exact words
- The user is exploring a topic conceptually rather than looking for a specific phrase
- The corpus uses technical jargon but the user writes in natural language

**Keyword search performs better when:**
- The user knows the exact title, author name, or paper ID they are looking for
- The query contains rare or domain-specific terms that have a precise meaning (e.g., model names like *GPT-4*, *BERT*)
- Recall precision is more important than conceptual recall

**Conclusion:** For open-ended exploration of an academic paper dataset, semantic search is generally superior.  
Keyword search is a reliable fallback for exact lookups. Hybrid search (Part 11) combines both strengths.

## Part 10 — Keyword vs Semantic Search Comparison

Using synonym query pairs to measure result overlap between the two methods.

In [19]:
synonym_pairs = [
    ('AI text generation',         'machine-written content synthesis'),
    ('deep learning image recognition', 'convolutional neural network visual classification'),
    ('reinforcement learning agent', 'reward-based policy optimisation'),
]

n = 10
print(f'Overlap comparison (top-{n} results)\n')

for q1, q2 in synonym_pairs:
    kw1  = set(keyword_search(df,  q1, n_results=n)['paper_id'].astype(str))
    kw2  = set(keyword_search(df,  q2, n_results=n)['paper_id'].astype(str))
    sem1 = set(semantic_search(collection, q1, n_results=n)['paper_id'].astype(str))
    sem2 = set(semantic_search(collection, q2, n_results=n)['paper_id'].astype(str))

    kw_overlap  = len(kw1  & kw2)
    sem_overlap = len(sem1 & sem2)

    print(f'  Pair: "{q1}"')
    print(f'        "{q2}"')
    print(f'    Keyword overlap:  {kw_overlap}/{n}')
    print(f'    Semantic overlap: {sem_overlap}/{n}')
    print()

2026-05-13 11:21:48,477 - INFO - [SearchEngine] Keyword search 'AI text generation' → 0 results
2026-05-13 11:21:48,484 - INFO - [SearchEngine] Keyword search 'machine-written content synthesis' → 0 results
2026-05-13 11:21:48,510 - INFO - [Embedder] Generated embeddings: shape=(1, 384)
2026-05-13 11:21:48,523 - INFO - [SearchEngine] Semantic search 'AI text generation' → 10 results


Overlap comparison (top-10 results)



2026-05-13 11:21:48,718 - INFO - [Embedder] Generated embeddings: shape=(1, 384)
2026-05-13 11:21:48,730 - INFO - [SearchEngine] Semantic search 'machine-written content synthesis' → 10 results
2026-05-13 11:21:48,743 - INFO - [SearchEngine] Keyword search 'deep learning image recognition' → 0 results
2026-05-13 11:21:48,756 - INFO - [SearchEngine] Keyword search 'convolutional neural network visual classification' → 0 results
2026-05-13 11:21:48,905 - INFO - [Embedder] Generated embeddings: shape=(1, 384)
2026-05-13 11:21:48,922 - INFO - [SearchEngine] Semantic search 'deep learning image recognition' → 10 results


  Pair: "AI text generation"
        "machine-written content synthesis"
    Keyword overlap:  0/10
    Semantic overlap: 9/10



2026-05-13 11:21:49,031 - INFO - [Embedder] Generated embeddings: shape=(1, 384)
2026-05-13 11:21:49,045 - INFO - [SearchEngine] Semantic search 'convolutional neural network visual classification' → 10 results
2026-05-13 11:21:49,059 - INFO - [SearchEngine] Keyword search 'reinforcement learning agent' → 0 results
2026-05-13 11:21:49,075 - INFO - [SearchEngine] Keyword search 'reward-based policy optimisation' → 0 results
2026-05-13 11:21:49,142 - INFO - [Embedder] Generated embeddings: shape=(1, 384)
2026-05-13 11:21:49,153 - INFO - [SearchEngine] Semantic search 'reinforcement learning agent' → 10 results
2026-05-13 11:21:49,191 - INFO - [Embedder] Generated embeddings: shape=(1, 384)
2026-05-13 11:21:49,214 - INFO - [SearchEngine] Semantic search 'reward-based policy optimisation' → 10 results


  Pair: "deep learning image recognition"
        "convolutional neural network visual classification"
    Keyword overlap:  0/10
    Semantic overlap: 6/10

  Pair: "reinforcement learning agent"
        "reward-based policy optimisation"
    Keyword overlap:  0/10
    Semantic overlap: 3/10



**Observation:** Semantic search consistently shows higher overlap between synonym pairs.  
Keyword search struggles because the two query strings share few exact words.  
This confirms that semantic search handles paraphrasing and synonyms more reliably than keyword matching.

## Part 11 — Hybrid Search

Hybrid search combines the ranked lists from keyword and semantic search using **Reciprocal Rank Fusion (RRF)**:

$$\text{score}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}, \quad k = 60$$

Documents that rank highly in *both* lists get the highest combined scores.

In [20]:
test_query = 'large language model fine-tuning'
n = 5

kw_res  = keyword_search(df, test_query, n_results=n)
sem_res = semantic_search(collection, test_query, n_results=n)
hyb_res = hybrid_search(collection, df, test_query, n_results=n)

print(f'Query: "{test_query}"\n')

print('--- Keyword Search ---')
if kw_res.empty:
    print('  (no exact matches)')
else:
    print(kw_res[['paper_id','title','primary_category']].to_string(index=False))

print('\n--- Semantic Search ---')
print(sem_res[['paper_id','title','similarity']].to_string(index=False))

print('\n--- Hybrid Search (RRF) ---')
print(hyb_res[['paper_id','title','rrf_score']].to_string(index=False))

2026-05-13 11:21:54,823 - INFO - [SearchEngine] Keyword search 'large language model fine-tuning' → 0 results
2026-05-13 11:21:54,868 - INFO - [Embedder] Generated embeddings: shape=(1, 384)
2026-05-13 11:21:54,881 - INFO - [SearchEngine] Semantic search 'large language model fine-tuning' → 5 results
2026-05-13 11:21:54,891 - INFO - [SearchEngine] Keyword search 'large language model fine-tuning' → 0 results
2026-05-13 11:21:54,981 - INFO - [Embedder] Generated embeddings: shape=(1, 384)
2026-05-13 11:21:54,999 - INFO - [SearchEngine] Semantic search 'large language model fine-tuning' → 20 results
2026-05-13 11:21:55,076 - INFO - [HybridSearch] 'large language model fine-tuning' → 5 hybrid results (RRF k=60)


Query: "large language model fine-tuning"

--- Keyword Search ---
  (no exact matches)

--- Semantic Search ---
  paper_id                                                                                                                   title  similarity
 2604.2699 Ucsc-Nlp At Semeval-2026 Task 13: Multi-View Generalization And Diagnostic Analysis Of Machine-Generated Code Detection      0.4913
2604.16923                            Alignment Imprint: Zero-Shot Ai-Generated Text Detection Via Provable Preference Discrepancy      0.4895
2604.11796                C-Red: A Comprehensive Chinese Benchmark For Ai-Generated Text Detection Derived From Real-World Prompts      0.4640
2604.13692                         Breaking The Generator Barrier: Disentangled Representation For Generalizable Ai-Text Detection      0.4288
2605.00924                    Styleshield: Exposing The Fragility Of Aigc Detectors Through Continuous Controllable Style Transfer      0.4205

--- Hybrid Search (RRF) ---
 

**Observation:** Hybrid search surfaces results that appear in both lists and assigns them higher RRF scores.  
Results that are only in one list receive lower scores. This gives better precision than either method alone.

## Part 12 — Pipeline Integration

`run_pipeline.py` has been updated with **Step 5** which runs after the analytics step.  
It creates/reuses the ChromaDB collection and indexes all cleaned papers:

```python
# ── Step 5: embeddings & vector search ───────────────────────────
from src.embeddings.chroma_store import get_client, get_or_create_collection, add_papers
client     = get_client()
collection = get_or_create_collection(client)
added      = add_papers(collection, cleaned)
```

Subsequent pipeline runs skip already-indexed papers (upsert by paper_id).

## Part 13 — Analytical Questions

Using the completed semantic search system to answer three real questions about the ArXiv dataset.

### Question 1: What papers are most relevant to AI-generated content detection, and are they recent?

In [21]:
q1_results = query_collection(
    collection,
    query_texts=['detecting AI-generated content and machine-written text'],
    n_results=8,
    where={'published_year': {'$gte': 2024}},
)

print('Q1: Top papers on AI-generated content detection (2024+)\n')
for m, d in zip(q1_results[0]['metadatas'], q1_results[0]['distances']):
    print(f"  [{1-d:.3f}] [{m['published_year']}] {m['title'][:80]}")

2026-05-13 11:22:06,867 - INFO - [Embedder] Generated embeddings: shape=(1, 384)


Q1: Top papers on AI-generated content detection (2024+)

  [0.608] [2026] Rising Prevalence Of Detected Ai-Generated Text In Medical Literature: Longitudi
  [0.592] [2026] C-Red: A Comprehensive Chinese Benchmark For Ai-Generated Text Detection Derived
  [0.575] [2026] Is It Cake Or Is It Ai? A Systematic Review Of Human Uncertainty In Distinguishi
  [0.552] [2026] Reasoning-Aware Aigc Detection Via Alignment And Reinforcement
  [0.537] [2026] Ethics Testing: Proactive Identification Of Generative Ai System Harms
  [0.525] [2026] Many Ways To Be Fake: Benchmarking Fake News Detection Under Strategy-Driven Ai 
  [0.495] [2026] Tab-Audit: Detecting Ai-Fabricated Scientific Tables Via Multi-View Likelihood M
  [0.492] [2026] Breaking The Generator Barrier: Disentangled Representation For Generalizable Ai


**Interpretation:** The semantic search correctly surfaces papers about AI text detection published in 2024 or later,  
even when they use varied terminology (e.g. *watermarking*, *provenance*, *AIGC*).  
This shows the dataset is actively growing in this research area — relevant for understanding current trends.

### Question 2: Which highly-cited papers focus on efficient training of deep learning models?

In [22]:
q2_results = query_collection(
    collection,
    query_texts=['efficient training techniques for deep neural networks'],
    n_results=8,
    where={'citation_count': {'$gte': 50}},
)

print('Q2: Highly-cited papers on efficient deep learning training (citations >= 50)\n')
for m, d in zip(q2_results[0]['metadatas'], q2_results[0]['distances']):
    print(f"  [{1-d:.3f}] [cit={m['citation_count']}] {m['title'][:75]}")

2026-05-13 11:22:10,048 - INFO - [Embedder] Generated embeddings: shape=(1, 384)


Q2: Highly-cited papers on efficient deep learning training (citations >= 50)

  [0.361] [cit=77] From Redaction To Restoration: Deep Learning For Medical Image Anonymizatio
  [0.331] [cit=114] Single-Cycle Multidirectional Eog Classification Faster Than Human Reaction
  [0.317] [cit=110] Beyond Semantics: Uncovering The Physics Of Fakes Via Universal Physical De
  [0.302] [cit=97] Combating Pattern And Content Bias: Adversarial Feature Learning For Genera
  [0.287] [cit=81] Styleshield: Exposing The Fragility Of Aigc Detectors Through Continuous Co
  [0.285] [cit=75] Latent Space Probing For Adult Content Detection In Video Generative Models
  [0.279] [cit=120] Layer Consistency Matters: Elegant Latent Transition Discrepancy For Genera
  [0.279] [cit=105] Genvideolens: Where Lvlms Fall Short In Ai-Generated Video Detection?


**Interpretation:** Filtering by high citation count narrows results to established, influential papers.  
The combination of semantic search + citation filter is useful for literature review:  
it surfaces foundational work on efficient training (quantisation, pruning, distillation, etc.).

### Question 3: How do papers on NLP compare to computer vision papers in terms of semantic similarity to a general AI safety query?

In [23]:
safety_query = 'ensuring safety and alignment of artificial intelligence systems'

nlp_results = query_collection(
    collection,
    query_texts=[safety_query],
    n_results=5,
    where={'primary_category': {'$in': ['cs.CL', 'cs.LG']}},
)

cv_results = query_collection(
    collection,
    query_texts=[safety_query],
    n_results=5,
    where={'primary_category': {'$in': ['cs.CV', 'eess.IV']}},
)

nlp_avg_sim = np.mean([1 - d for d in nlp_results[0]['distances']])
cv_avg_sim  = np.mean([1 - d for d in cv_results[0]['distances']])

print(f'Query: "{safety_query}"\n')
print(f'NLP/ML papers (cs.CL / cs.LG) — avg similarity: {nlp_avg_sim:.4f}')
for m, d in zip(nlp_results[0]['metadatas'], nlp_results[0]['distances']):
    print(f"  [{1-d:.3f}] [{m['primary_category']}] {m['title'][:70]}")

print(f'\nCV papers (cs.CV / eess.IV) — avg similarity: {cv_avg_sim:.4f}')
for m, d in zip(cv_results[0]['metadatas'], cv_results[0]['distances']):
    print(f"  [{1-d:.3f}] [{m['primary_category']}] {m['title'][:70]}")

winner = 'NLP/ML' if nlp_avg_sim > cv_avg_sim else 'Computer Vision'
print(f'\n=> {winner} papers are more semantically aligned with AI safety topics.')

2026-05-13 11:22:14,419 - INFO - [Embedder] Generated embeddings: shape=(1, 384)
2026-05-13 11:22:14,672 - INFO - [Embedder] Generated embeddings: shape=(1, 384)


Query: "ensuring safety and alignment of artificial intelligence systems"

NLP/ML papers (cs.CL / cs.LG) — avg similarity: 0.2442
  [0.298] [cs.CL] Tab-Audit: Detecting Ai-Fabricated Scientific Tables Via Multi-View Li
  [0.250] [cs.CL] C-Red: A Comprehensive Chinese Benchmark For Ai-Generated Text Detecti
  [0.239] [cs.CL] Dia-Harm: Dialectal Disparities In Harmful Content Detection Across 50
  [0.230] [cs.CL] Many Ways To Be Fake: Benchmarking Fake News Detection Under Strategy-
  [0.204] [cs.CL] Breaking The Generator Barrier: Disentangled Representation For Genera

CV papers (cs.CV / eess.IV) — avg similarity: 0.2822
  [0.342] [cs.CV] Agentfox: Llm Agent-Guided Fusion With Explainability For Ai-Generated
  [0.284] [cs.CV] Quality-Aware Calibration For Ai-Generated Image Detection In The Wild
  [0.270] [cs.CV] Generalizable Detection Of Ai Generated Images With Large Models And F
  [0.258] [cs.CV] Dymapia: A Multi-Domain Framework For Detecting Ai-Based Video Manipul
  [0.257] [cs.C

**Interpretation:** This comparison reveals which research community in the dataset publishes work  
more semantically related to AI safety. NLP/ML papers typically score higher on this query because  
alignment, hallucination, and bias research are core topics in language model research.  
Computer vision papers focus more on visual perception tasks, which are less semantically connected to AI safety discourse.